# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishalseries7030/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.

Unit of analysis:** One row represents one content item in the anonymized content-refresh dataset. The dataset contains 30,000 rows and 30,000 unique `content_id` values; this is verified below.

Time window:** The available dataset schema does not establish a calendar-date range from the inspected fields. Columns such as `impressions_90d`, `clicks_90d`, and `pageviews_90d` indicate 90-day metrics, while `days_since_last_update` is a duration, not a date. The exact observation window must not be inferred from these fields alone.

In [6]:
print("Rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Missing content IDs:", int(df["content_id"].isna().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

assert df["content_id"].nunique() == len(df), (
    "content_id is not unique; revisit the unit of analysis."
)

Rows: 30000
Unique content IDs: 30000
Missing content IDs: 0
Duplicate rows: 0


In [5]:

!git clone https://github.com/vishalseries7030/flyrank-ml-internship.git /content/flyrank-ml-internship


fatal: destination path '/content/flyrank-ml-internship' already exists and is not an empty directory.


Repository exists: True

CSV files inside data folders:
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


## 2. Fields: feature / label / context / excluded


*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Features:** Candidate content attributes and pre-decision signals such as `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`, `word_count`, `char_count`, and eligible historical performance metrics. These are candidate features only; each must be checked for availability at prediction time.

**Label:** `trend_direction` can serve as an observed proxy target for exploratory analysis. It is not proof of future decline. The schema does not contain an `is_declining_label` column.

**Context:** `content_id` and `client_id` are used for joins, grouping, split design, and traceability—not as predictive features. `provider_used` and `model_used` describe how content was produced and should be treated as context unless a documented use case justifies otherwise.

**Excluded:** Identifiers (`content_id`, `client_id`) are excluded from model inputs to prevent memorization or leakage. Any post-decision fields, unavailable-at-decision-time metrics, private client details, and fields with unclear timing are excluded until their validity is established. `days_since_last_update` is a duration, not a calendar date.

**Validation rule:** Final feature selection depends on column definitions, missingness, target availability, and whether each field existed before the decision being supported.

In [7]:

# Inspect the actual schema and key fields before finalizing classifications.

print("Total columns:", len(df.columns))
print("\nAll columns:")
for col in df.columns:
    print("-", col)

print("\nKey field checks:")
for col in [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "days_since_last_update",
]:
    print(f"{col}: {'PRESENT' if col in df.columns else 'NOT PRESENT'}")

print("\nMissing values in key fields:")
for col in [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "days_since_last_update",
]:
    if col in df.columns:
        print(f"{col}: {int(df[col].isna().sum())} missing")


Total columns: 44

All columns:
- content_id
- client_id
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count
- provider_used
- model_used
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- age_tier
- age_tier_order
- days_since_last_update
- freshness_tier
- word_count_tier
- char_count_tier
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct
- impression_tier
- position_tier
- trend_direction
- trend_pct

Key field checks:
content_id: PRESENT
client_id: PRESENT
trend_direction: PRESENT
trend_pct: PRESENT
is_declining_label: NOT PRESENT
days_since_last_update: PRESENT

Missing values in key fields:
content_id: 0 missing
client_id: 0 missing
tr

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Verification approach:** The checks below validate row grain, identifier uniqueness, missingness, observed trend labels, and available time-related fields.

**Interpretation:** `trend_direction` is an observed trend field and may be used as a proxy target for exploratory analysis; it must not be described as verified future decline. `trend_pct` has missing values, so analyses using it must report the resulting sample size. Fields ending in `_90d` and `_last_30d` describe metric windows, but do not independently establish the dataset's overall calendar-date coverage.

**Data quality:** The initial inspection found 30,000 rows, 44 columns, 30,000 unique `content_id` values, no missing `content_id`, no missing `client_id`, no missing `trend_direction`, and 3,388 missing `trend_pct` values. The next code cell reproduces these checks.

In [8]:

# Reproducible data-contract checks

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique content IDs:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())
print("Duplicate rows:", int(df.duplicated().sum()))

print("\nMissing values:")
for col in ["content_id", "client_id", "trend_direction", "trend_pct"]:
    print(f"{col}: {int(df[col].isna().sum())}")

print("\nTrend direction counts:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nContent type counts:")
print(df["content_type"].value_counts(dropna=False))

print("\nDuration fields — descriptive statistics:")
for col in ["content_age_days", "days_since_last_update"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].describe())


Rows: 30000
Columns: 44
Unique content IDs: 30000
Unique clients: 32
Duplicate rows: 0

Missing values:
content_id: 0
client_id: 0
trend_direction: 0
trend_pct: 3388

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Content type counts:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

Duration fields — descriptive statistics:

content_age_days:
count    30000.00000
mean       256.16780
std        132.70793
min         90.00000
25%        132.00000
50%        236.00000
75%        333.00000
max        564.00000
Name: content_age_days, dtype: float64

days_since_last_update:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Coverage and representativeness:** The dataset contains 30,000 content items across 32 clients. Content types are imbalanced: 27,207 keyword articles, 2,096 Feedly articles, and 697 comparison articles. Overall metrics may therefore primarily reflect keyword articles and should not be assumed to generalize to the smaller groups.

**Missing data:** `trend_pct` is missing for 3,388 rows. `avg_position` contains zero values that may represent missing or unavailable ranking data rather than a genuine rank of zero; these values must be checked before use.

**Target limitations:** `trend_direction` is an observed category, not proof of future decline or causal impact. Any binary target derived from it is a proxy and must be clearly described as such.

**Time and leakage limitations:** The dataset contains duration fields and rolling-window metrics, but these alone do not establish the full calendar-date coverage or prove that every feature was available at decision time. Potentially post-outcome or overlapping-window fields must be reviewed for leakage.

**Generalization:** Results from a client-grouped split or one dominant content type may not represent all clients, time periods, or content types. The dataset alone cannot establish that recommended actions will cause performance improvements.

**Privacy:** Do not publish client names, domains, private queries, credentials, or raw sensitive exports. Use anonymized aggregate results only.

In [9]:

# Final data-limit checks

print("Content type proportions (%):")
print(
    (df["content_type"].value_counts(normalize=True, dropna=False) * 100)
    .round(2)
)

print("\ntrend_pct missing:")
print(int(df["trend_pct"].isna().sum()), "of", len(df))

print("\navg_position == 0:")
if "avg_position" in df.columns:
    print(int((df["avg_position"] == 0).sum()))
else:
    print("Column not present")

print("\nTrend categories:")
print(df["trend_direction"].value_counts(dropna=False).to_dict())

print("\nCalendar date columns:")
date_cols = [
    c for c in df.columns
    if "date" in c.lower() or "timestamp" in c.lower()
]
print(date_cols if date_cols else "No explicit calendar date/timestamp columns found")

print("\nReminder: rolling-window metrics and duration fields do not prove feature availability at decision time.")


Content type proportions (%):
content_type
keyword article       90.69
feedly article         6.99
comparison article     2.32
Name: proportion, dtype: float64

trend_pct missing:
3388 of 30000

avg_position == 0:
1205

Trend categories:
{'down': 16262, 'stable': 5962, 'up': 4388, 'new': 2236, 'flat': 1152}

Calendar date columns:
['days_since_last_update']

Reminder: rolling-window metrics and duration fields do not prove feature availability at decision time.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.